In [1]:
%load_ext autoreload
%autoreload 2
#%export SETUPTOOLS_USE_DISTUTILS=stdlib
#%cd /home/felix/Desktop/AA_Uni/Projektarbeit/test/Genesis-Dog-Walking/test

import genesis as gs
import logging
import utils
import torch
import os

#if torch.cuda.is_available():
#    gs.init(logging_level=logging.WARNING, backend=gs.gpu)
#else:
gs.init(logging_level=logging.WARNING, backend=gs.gpu)

from buffer import Buffer
from network import Network
from make_environment import Go2WalkingEnv
from reward import Rewards

path = os.getcwd()



[I 03/18/26 07:52:27.410 3947756] [shell.py:_shell_pop_print@25] Graphical python shell detected, using wrapped sys.stdout
2026-03-18 07:52:29.866 Python[85278:3947756] ApplePersistenceIgnoreState: Existing state will not be touched. New state will be written to /var/folders/3y/snyjd7qd59d_gxq1x0y5gywh0000gn/T/org.python.python.savedState


In [2]:
reward_fn = Rewards()
num_envs = 512
max_steps = 100
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

env = Go2WalkingEnv(
    num_envs=num_envs,
    device=device,
    show_viewer=False,
    use_terrain=False,  
    episode_length_s=20.0,
    reward_fn=reward_fn,
    min_up_dot=0.05
)

[Genesis] [07:52:31] [WARNING] Viewer option 'n_rendered_envs' is deprecated and will be removed in future release. Please use 'rendered_envs_idx' instead.
[Genesis] [07:52:37] [WARNING] Neutral robot position (qpos0) exceeds joint limits.


In [3]:
env.set_commands(lin_vel_x=1.0, lin_vel_y=0.0, ang_vel_yaw=0.0)
policy = Network(
    num_outputs=env.num_actions,
    num_inputs=env.num_obs,
    gamma=0.99,
    lmbda=0.0,
    epsilon=0.1,
)
buffer = Buffer(
    num_envs=num_envs,
    obs_dim=env.num_obs,
    act_dim=env.num_actions,
    max_length=max_steps,
    device=device
)

In [4]:
import torch
policy.apply(lambda m: torch.nn.init.xavier_uniform_(m.weight) if hasattr(m, 'weight') else None)

Network(
  (shared): Sequential(
    (0): Linear(in_features=48, out_features=512, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ELU(alpha=1.0)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ELU(alpha=1.0)
  )
  (actor): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=64, out_features=32, bias=True)
  )
  (actor_mean): Linear(in_features=32, out_features=12, bias=True)
  (critic): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [5]:
def adjust_scales(update):
    if update < 15:
        scales = { # No walking reward, High termiantion penaltym, focus on not falling and not moving sideways
            "tracking_lin_vel_x": 0.0,
            "tracking_ang_vel": 0.2,
            "x_progress": 0.0,
            "lin_vel_z": -0.1,
            "lin_vel_y": -0.05,
            "action_rate": -0.002,
            "similar_to_default": -1.0,
            "termination": -6.0,
            "sideway_movement": -0.2,
            "base_height": 0.1,
        }
    elif update < 50:
        scales = {
            "tracking_lin_vel_x": 0.0,
            "tracking_ang_vel": 0.0,
            "x_progress": 0.0,
            "lin_vel_z": -0.2,
            "lin_vel_y": -0.1,
            "action_rate": -0.002,
            "similar_to_default": -0.5,
            "termination": -6.0,
            "sideway_movement": -0.1,
            "base_height": 0.1,
        }

    elif update < 75:
        scales = {
            "tracking_lin_vel_x": 1.0,
            "tracking_ang_vel": 1.0,
            #"x_progress": 0.5,
            "lin_vel_z": -1.0,
            "lin_vel_y": -5.0,
            "action_rate": -0.005,
            "similar_to_default": -0.1,
            #"termination": -4.0,
            "sideway_movement": -1.0,
            #"base_height": 0.1,
        }"""
    elif update < 150:
        scales = {
            "tracking_lin_vel_x": 1.5,
            "tracking_ang_vel": 0.5,
            "x_progress": 0.2,
            "lin_vel_z": -0.2,
            "lin_vel_y": -0.1,
            "action_rate": -0.005,
            "similar_to_default": -0.02,
            "termination": -5.0,
            "sideway_movement": -0.1,
            "base_height": 0.1,
        }
    else:
        scales = {
            "tracking_lin_vel_x": 2.5,
            "tracking_ang_vel": 0.5,
            "x_progress": 0.2,
            "lin_vel_z": -0.2,
            "lin_vel_y": -0.1,
            "action_rate": -0.01,
            "similar_to_default": -0.02,
            "termination": -5.0,
            "sideway_movement": -0.1,
            "base_height": 0.1,
        }"""
    reward_fn.scales = scales

In [6]:
optim = torch.optim.Adam(policy.parameters(), lr=5e-4, eps=1e-8)
#torch.autograd.set_detect_anomaly(True)
num_updates = 1000
steps_per_update = 128
update_epochs = 5
minibatch_size = 512
start_update = 150
#max_length =

buffer = Buffer(
    num_envs=num_envs,
    obs_dim=env.num_obs,
    act_dim=env.num_actions,
    max_length=steps_per_update,
    device=device
)

for i in range(start_update, num_updates):
    adjust_scales(i)
    print(f"Running Sim: i: {i}")
    with torch.no_grad():
        buffer.reset()
        obs = env.reset()
        buffer.init_obs(obs, policy.get_value(obs))
        for step in range(steps_per_update):
            obs = obs.to(device)
            actions, value = policy.get_actions(obs)
            log_probs, log_probs_value, entropy = policy.compute_log_probs(obs, actions)
            next_obs , reward, done, info = env.step(actions)
            buffer.add_step(next_obs, actions, log_probs, reward, done, value)

            obs = next_obs

            if done.any():
                obs = env.reset()

        buffer.compute_returns_and_advantages(gamma=0.99, lmbda=0.95)
    print(f"Running Epochs: i: {i}, Avg_reward: {buffer.rewards.mean():.3f}")
    for epoch in range(update_epochs):
        batch = buffer.get_batch(minibatch_size)

        log_probs_new, values_new, entropy = policy.compute_log_probs(batch['obs'], batch['actions'])
        #print(batch)
        critic_loss, actor_loss = policy.compute_loss(states= batch['obs'],
                                                      actions= batch['actions'],
                                                      advantages=batch['advantages'],
                                                      critic_targets=batch['values'],
                                                      log_probs_old=batch['log_probs'],
                                                      returns = batch['returns'],)
        entropy_loss = -0.01 * entropy.mean()

        loss = actor_loss + 0.5 * critic_loss + entropy_loss

        optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 0.5) # gradient clipping
        optim.step()

    if i % 10 == 0:
        print(utils.save_checkpoint(
            path=f"{path}/checkpoints/go2_update_{i}.pt",
            policy=policy,
            optim=optim,
            update=i,
            avg_rew=buffer.rewards.mean().item()))
        print(utils.make_eval_video(
            env=env,
            policy=policy,
            filename=f"{path}/video/eval_update_{i}.mp4",
            eval_steps=600,
        ))


Running Sim: i: 150
Running Epochs: i: 150, Avg_reward: 0.720
None


UNSUPPORTED (log once): POSSIBLE ISSUE: unit 4 GLD_TEXTURE_INDEX_CUBE_MAP is unloadable and bound to sampler type (Float) - using zero texture because texture unloadable


{'video_path': '/Users/felix/PycharmProjects/Genesis-Dog-Walking/test/video/eval_update_150.mp4', 'episode_reward': 376.5046936273575, 'steps': 600}
Running Sim: i: 151
Running Epochs: i: 151, Avg_reward: 0.752
Running Sim: i: 152
Running Epochs: i: 152, Avg_reward: 0.723
Running Sim: i: 153
Running Epochs: i: 153, Avg_reward: 0.714
Running Sim: i: 154
Running Epochs: i: 154, Avg_reward: 0.711
Running Sim: i: 155


KeyboardInterrupt: 

In [ ]:
utils.save_checkpoint(
        path=f"/Users/felix/PycharmProjects/Genesis-Dog-Walking/test/go2_update_Final.pt",
        policy=policy,
        optim=optim,
        update=100,
        avg_rew=buffer.rewards.mean().item(),
    )
